## Section 0: Imports & Setup

In [1]:
#!/usr/bin/env python
# coding: utf-8
"""
Experiment 2 — Statistical Analysis (v3, fixed for pymer4 0.9.2 API)
=========================================
Preprocessing
-------------
1. order:  rescaled  1/2 → 0/1  (subtract 1, stored as order_c)
2. rating: centred only (subtract grand mean), NOT z-scored  →  rating_cen

Model structure (per outcome)
------------------------------
  Null:        outcome ~ 1 + RE
  Additive:    outcome ~ IVs + covariates + RE
  Interactive: outcome ~ IVs + all 2-way interactions + covariates + RE

Random effects
--------------
  Max: (1 + source_test | trace)   ← tested on each interactive model
  Red: (1 | trace)                  ← fallback if max is singular
  RE selection: LRT (interactive_max vs interactive_red)

Model comparisons  (LRT throughout)
------------------------------------
  Step 1 — RE:  interactive_max vs interactive_red
  Step 2 — FE:  null vs additive        (same RE as step 1 winner)
  Step 3 — FE:  additive vs interactive (same RE)

Outcomes
--------
  1.   RH (perceived)     binomial GLMM   logit
  2.   Accuracy           binomial GLMM   logit   all trials
  2P.  Accuracy-P         binomial GLMM   logit   perceived + RH predictor
  3.   Rating             Gaussian LMM    identity all trials
  3P.  Rating-P           Gaussian LMM    identity perceived + RH predictor
  4.   Confidence         Gaussian LMM    identity all trials
  4P.  Confidence-P       Gaussian LMM    identity perceived + RH predictor
  5.   Gamma              Gaussian LMM    identity trace-level
  5P.  Gamma-P            Gaussian LMM    identity trace-level, perceived + rh_mean

Covariates (included in additive & interactive, not null)
---------------------------------------------------------
  rating_cen  — mean-centred associative rating
  order_c     — presentation order (0/1)
  model       — LLM architecture (6 levels)

Post-hoc (on best interactive model per outcome)
-------------------------------------------------
  • Pairwise emmeans: source_test, fb_exp, setsize, model
  • Interaction contrasts: source_test × fb_exp  (H3)
  • Covariate slopes: rating_cen, order_c  (extracted from FE table)
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. Imports & setup
# ─────────────────────────────────────────────────────────────────────────────
import re
import gc
import warnings, os
import numpy as np
import pandas as pd
import polars as pl
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — no GUI, no segfault
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro
from IPython.display import display, Markdown

import rmllm
from rmllm import gamma as gamma_mod

# pymer4 ≥ 0.9 API
from pymer4.models import lmer, glmer, compare

# rpy2 — DHARMa diagnostics only
# pandas2ri.activate() removed: deprecated/raises in rpy2 ≥ 3.5
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
np.random.seed(42)


def _r_pkg(name):
    try:
        return importr(name)
    except Exception:
        warnings.warn(f"R package '{name}' not available — some diagnostics skipped.")
        return None

_DHARMa  = _r_pkg("DHARMa")
_stats_r = importr("stats")
_base_r  = importr("base")
# FIX #7: import lme4 for isSingular check
_lme4    = importr("lme4")

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams.update({"font.family": "serif",
                     "font.serif":  ["Times New Roman", "DejaVu Serif"]})
FB_PALETTE = {"True": "#2196F3", "False": "#FF9800"}
PLOT_DIR   = "."

data_dir = rmllm.config.PROCESSED_DATA_DIR
print("Environment ready.")

2026-06-18 11:24:00.314 | INFO     | rmllm.config:<module>:11 - PROJ_ROOT path is: <PROJECT_ROOT>


Environment ready.


## Section 1: Data Loading & Preprocessing

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Data loading and preprocessing
# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_csv(data_dir / "exp2_trial_data.csv")

# ── Numeric coercions ─────────────────────────────────────────────────────────
for col in ["rating", "confidence", "order", "trial_compliance", "accuracy"]:
    df[f"{col}_num"] = pd.to_numeric(df.get(col), errors="coerce")

df["trial_compliance"]   = df["trial_compliance_num"]
df["accuracy"]           = df["accuracy_num"]
df["read_hallucination"] = 1 - df["trial_compliance_num"]

# ── Preprocessing 1: order rescaled 1/2 → 0/1 ────────────────────────────────
df["order_c"] = df["order_num"] - 1
print("order_c values:", sorted(df["order_c"].dropna().unique()))

# ── Preprocessing 2: rating centred only (not z-scored) ──────────────────────
_rating_mean  = df["rating_num"].mean()
df["rating_cen"] = df["rating_num"] - _rating_mean
print(f"rating_cen: mean={df['rating_cen'].mean():.4f}  sd={df['rating_cen'].std():.4f}")

# ── Observation-level ID (OLRE models) ───────────────────────────────────────
df["obs_id"] = np.arange(len(df)).astype(str)

# ── String cast for pymer4 factor handling ────────────────────────────────────
for col in ["setsize", "fb_exp", "model", "source_test", "obs_id"]:
    if col in df.columns:
        df[col] = df[col].astype(str)

# ── Subsets ───────────────────────────────────────────────────────────────────
df_perc = df[df["source_test"] == "test:perceived"].copy()
df_imag = df[df["source_test"] == "test:imagined"].copy()

grp_between = ["model", "setsize", "fb_exp"]
grp_within  = ["source_test"]
sim_id      = "trace"

print(f"\nAll trials  : {len(df):,}   Traces: {df[sim_id].nunique():,}")
print(f"Perceived   : {len(df_perc):,}  (RH rate: {df_perc['read_hallucination'].mean():.3f})")
print(f"Imagined    : {len(df_imag):,}  (RH rate: {df_imag['read_hallucination'].mean():.3f})")
display(df[["read_hallucination","accuracy","rating_cen","confidence_num","order_c"
            ]].describe().round(3))

order_c values: [np.int64(0), np.int64(1)]
rating_cen: mean=0.0000  sd=20.2663

All trials  : 72,000   Traces: 4,800
Perceived   : 36,000  (RH rate: 0.406)
Imagined    : 36,000  (RH rate: 0.000)


,read_hallucination,accuracy,rating_cen,confidence_num,order_c
count,72000.000,72000.000,72000.000,72000.000,72000.0
mean,0.203,0.564,0.000,5.114,0.5
std,0.402,0.496,20.266,1.407,0.5
min,0.000,0.000,-79.382,1.000,0.0
25%,0.000,0.000,-4.382,5.000,0.0
50%,0.000,1.000,5.618,6.000,0.5
75%,0.000,1.000,10.618,6.000,1.0
max,1.000,1.000,20.618,6.000,1.0


## Section 2: Base Helper Functions

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. Base helper functions
# ─────────────────────────────────────────────────────────────────────────────

def _to_polars(data):
    pf = pl.from_pandas(data) if isinstance(data, pd.DataFrame) else data
    cat_cols = ["source_test", "setsize", "fb_exp", "model", "obs_id"]
    casts = {c: pl.String for c in cat_cols if c in pf.columns}
    return pf.cast(casts) if casts else pf


def _setup_factors(m):
    cols = m.data.columns if hasattr(m.data, "columns") else []
    factors = {}
    if "source_test" in cols:
        factors["source_test"] = ["test:perceived", "test:imagined"]
    if "setsize"     in cols:
        factors["setsize"]     = ["20", "40"]
    if "fb_exp"      in cols:
        factors["fb_exp"]      = ["False", "True"]
    if "model"       in cols:
        factors["model"]       = ["Gemma3:12b", "Gemma3:12b-QAT",
                                   "Gemma3:27b", "Gemma3:27b-QAT",
                                   "Llama4:16x17b", "Llama3.3:70b"]
    if factors:
        m.set_factors(factors)


# Primary optimizer: bobyqa (robust for both LMM and GLMM)
# Secondary optimizer: nloptwrap (tried automatically on convergence failure)
# Tertiary optimizer: Nelder_Mead (tried if nloptwrap also fails)
_BOBYQA_G      = "glmerControl(optimizer='bobyqa',      optCtrl=list(maxfun=500000))"
_NLOPTWRAP_G   = "glmerControl(optimizer='nloptwrap',   optCtrl=list(maxfun=500000))"
_NELDER_MEAD_G = "glmerControl(optimizer='Nelder_Mead', optCtrl=list(maxfun=500000))"
_BOBYQA_L      =  "lmerControl(optimizer='bobyqa',      optCtrl=list(maxfun=500000))"
_NLOPTWRAP_L   =  "lmerControl(optimizer='nloptwrap',   optCtrl=list(maxfun=500000))"
_NELDER_MEAD_L =  "lmerControl(optimizer='Nelder_Mead', optCtrl=list(maxfun=500000))"


# FIX #1: Use lme4.isSingular(m.r_model) instead of m.warnings
def is_singular(m):
    """Return True if the fitted model triggered an isSingular warning."""
    try:
        return bool(_lme4.isSingular(m.r_model)[0])
    except Exception:
        return False


# FIX #2: Use m.convergence_status string instead of m.warnings
def _has_conv_failure(m):
    """Return True if bobyqa hit maxfun or otherwise failed to converge.
    Distinct from isSingular: convergence failure means the optimiser never
    found a minimum; isSingular means it found one on a boundary.

    In pymer4 0.9.2, convergence_status is a string like:
      'Convergence status\n: [1] FALSE\nattr(,"gradient")\n[1] NA\n'        (OK)
      'Convergence status\n: [1] FALSE\nattr(,"gradient")\n[1] 0.031\n'     (FAILED)
    The '[1] FALSE' means isSingular=FALSE (not singular) — it is always present.
    The real failure signal is a non-NA gradient value exceeding the threshold.
    """
    conv_str = str(getattr(m, "convergence_status", ""))
    # Check for large gradient (> 0.002 threshold); skip if gradient is NA
    try:
        m_grad = re.search(
            r'gradient[^\n]*\n\[1\]\s+(\S+)',
            conv_str,
            re.DOTALL
        )
        if m_grad:
            val_str = m_grad.group(1)
            if val_str.upper() != "NA":
                grad = float(val_str)
                return grad > 0.002
    except Exception:
        pass
    return False


# FIX #6: _refit_nloptwrap with Nelder_Mead third attempt
def _refit_nloptwrap(formula, data, family, label):
    """Re-fit with nloptwrap when bobyqa convergence fails.
    Falls back to Nelder_Mead if nloptwrap also fails. Returns model or None."""
    ctrl_nlopt  = _NLOPTWRAP_G   if family == "binomial" else _NLOPTWRAP_L
    ctrl_nelder = _NELDER_MEAD_G if family == "binomial" else _NELDER_MEAD_L

    # Attempt 1: nloptwrap
    try:
        if family == "binomial":
            m2 = glmer(formula, data=_to_polars(data), family="binomial")
        else:
            m2 = lmer(formula, data=_to_polars(data))
        _setup_factors(m2)
        m2.fit(control=ctrl_nlopt)
        print(f"  ✓  [{label}] nloptwrap converged")
        if not _has_conv_failure(m2):
            return m2
        print(f"  ⚠  [{label}] nloptwrap still did not converge → trying Nelder_Mead")
    except Exception as e:
        print(f"  ✗  [{label}] nloptwrap failed: {e} → trying Nelder_Mead")

    # Attempt 2: Nelder_Mead
    try:
        if family == "binomial":
            m3 = glmer(formula, data=_to_polars(data), family="binomial")
        else:
            m3 = lmer(formula, data=_to_polars(data))
        _setup_factors(m3)
        m3.fit(control=ctrl_nelder)
        print(f"  ✓  [{label}] Nelder_Mead converged")
        return m3
    except Exception as e:
        print(f"  ✗  [{label}] Nelder_Mead also failed: {e}")
        return None


def fit_glmer(formula, data, label="", fallback_formula=None):
    m = glmer(formula, data=_to_polars(data), family="binomial")
    _setup_factors(m)
    print(f"[{label}] Fitting GLMM: {formula[:80]}")
    m.fit(control=_BOBYQA_G)

    # Step 1 — convergence failure → retry with nloptwrap
    if _has_conv_failure(m):
        print(f"  ⚠  [{label}] bobyqa did not converge → trying nloptwrap")
        m2 = _refit_nloptwrap(formula, data, "binomial", label)
        if m2 is not None and not _has_conv_failure(m2):
            m = m2

    # Step 2 — singular RE → fallback formula (simpler RE structure)
    if is_singular(m) and fallback_formula:
        print(f"  ⚠  [{label}] isSingular → RE fallback: {fallback_formula[-40:]}")
        m = glmer(fallback_formula, data=_to_polars(data), family="binomial")
        _setup_factors(m)
        m.fit(control=_BOBYQA_G)
        if _has_conv_failure(m):
            m2 = _refit_nloptwrap(fallback_formula, data, "binomial", label + " fallback")
            if m2 is not None:
                m = m2

    _print_stats(m, label)
    return m


def fit_lmm(formula, data, label="", fallback_formula=None):
    m = lmer(formula, data=_to_polars(data))
    _setup_factors(m)
    print(f"[{label}] Fitting LMM: {formula[:80]}")
    m.fit(control=_BOBYQA_L)

    # Step 1 — convergence failure → retry with nloptwrap
    if _has_conv_failure(m):
        print(f"  ⚠  [{label}] bobyqa did not converge → trying nloptwrap")
        m2 = _refit_nloptwrap(formula, data, "gaussian", label)
        if m2 is not None and not _has_conv_failure(m2):
            m = m2

    # Step 2 — singular RE → fallback formula
    if is_singular(m) and fallback_formula:
        print(f"  ⚠  [{label}] isSingular → RE fallback: {fallback_formula[-40:]}")
        m = lmer(fallback_formula, data=_to_polars(data))
        _setup_factors(m)
        m.fit(control=_BOBYQA_L)
        if _has_conv_failure(m):
            m2 = _refit_nloptwrap(fallback_formula, data, "gaussian", label + " fallback")
            if m2 is not None:
                m = m2

    _print_stats(m, label)
    return m


# FIX #3: Use m.result_fit_stats Polars DataFrame
def _print_stats(m, label):
    display(Markdown(f"**{label} — Fit statistics**"))
    try:
        stats_df = m.result_fit_stats.to_pandas()
        display(stats_df.round(4))
    except Exception:
        # Fallback: try scalar attributes for older API compat
        stats_row = {}
        for attr, key in [("AIC", "AIC"), ("BIC", "BIC"), ("logLike", "logLike"),
                          ("Deviance", "Deviance"), ("Df.resid", "Df.resid")]:
            val = getattr(m, attr, None)
            if val is not None:
                try:
                    stats_row[key] = round(float(val), 4)
                except (TypeError, ValueError):
                    pass
        if stats_row:
            display(pd.DataFrame([stats_row]))
    conv_str = str(getattr(m, "convergence_status", ""))
    if conv_str.strip():
        print(f"  convergence_status: {conv_str[:200]}")


# FIX #9: result_fit is Polars DataFrame; p-value column is p_value
def show_fe(m, label="", exponentiate=False):
    tbl = m.result_fit.to_pandas()
    p_col = next((c for c in tbl.columns
                  if c.lower() in ("p_value", "p", "pr(>|z|)", "pr(>|t|)")), None)
    if p_col:
        tbl["sig"] = tbl[p_col].map(
            lambda p: "***" if pd.notnull(p) and p < .001
                      else "**"  if pd.notnull(p) and p < .01
                      else "*"   if pd.notnull(p) and p < .05
                      else "")
    if exponentiate:
        for raw, name in [("estimate","OR"), ("Estimate","OR"),
                          ("conf_low","OR_lo"), ("2.5 %","OR_lo"), ("lower","OR_lo"),
                          ("conf_high","OR_hi"), ("97.5 %","OR_hi"), ("upper","OR_hi")]:
            if raw in tbl.columns:
                tbl[name] = np.exp(tbl[raw])
    display(Markdown(f"**{label} — Fixed effects**"))
    display(tbl.round(4))
    return tbl


# FIX #8: p_value column added explicitly to p_col lookup
def run_anova(m, label=""):
    try:
        m.anova()
        tbl   = m.result_anova.to_pandas()
        p_col = next((c for c in tbl.columns
                      if c.lower() in ("p_value", "p","p.value","pr(>f)","pr(>chisq)")), None)
        if p_col:
            tbl["sig"] = tbl[p_col].map(
                lambda p: "***" if pd.notnull(p) and p < .001
                          else "**"  if pd.notnull(p) and p < .01
                          else "*"   if pd.notnull(p) and p < .05
                          else "")
        display(Markdown(f"**{label} — Type-III ANOVA (Satterthwaite)**"))
        display(tbl.round(4))
        return tbl
    except Exception as e:
        print(f"  [ANOVA — {e}]")
        return None


# FIX #11: compare() returns Polars DataFrame; handle Pr(>Chisq) column
def lrt(m_full, m_red, label=""):
    display(Markdown(f"**{label} — LRT**"))
    try:
        res = compare(m_red, m_full, test="LRT", as_dataframe=True)
        tbl = res.to_pandas() if isinstance(res, pl.DataFrame) else res
        p_col = next((c for c in tbl.columns
                      if "p" in c.lower() and c.lower() != "npar"), None)
        if p_col:
            tbl["sig"] = tbl[p_col].map(
                lambda p: "***" if pd.notnull(p) and p < .001
                          else "**"  if pd.notnull(p) and p < .01
                          else "*"   if pd.notnull(p) and p < .05
                          else "")
        display(tbl.round(4))
        return tbl
    except Exception as e:
        print(f"  [LRT — {e}]")
        return None


def vif(m, label=""):
    display(Markdown(f"**{label} — VIF**"))
    try:
        v = m.vif()
        v = v.to_pandas() if isinstance(v, pl.DataFrame) else v
        high = v[v.iloc[:, 1] > 5] if v.shape[1] > 1 else pd.DataFrame()
        print("  ⚠  High VIF:" if not high.empty else "  ✓ All VIF < 5",
              list(high.iloc[:,0]) if not high.empty else "")
        display(v.round(3))
        return v
    except Exception as e:
        print(f"  [VIF — {e}]")
        return None


# FIX #10: emmeans() returns Polars DataFrame; handle p.value or p_value
def pairwise(m, var, by=None, adjust="fdr", label=""):
    desc = f"{var}" + (f" | {by}" if by else "")
    display(Markdown(f"**{label} — Pairwise emmeans: {desc} [{adjust}]**"))
    try:
        res = m.emmeans(marginal_var=var, by=by, contrasts="pairwise", p_adjust=adjust)
        if isinstance(res, pl.DataFrame):
            res = res.to_pandas()
        # FIX: find p-value column (p.value or p_value), excluding adjusted columns
        p_col = next(
            (c for c in res.columns
             if "p" in c.lower() and "adjust" not in c.lower()
             and c.lower() not in ("npar",)),
            None
        )
        if p_col:
            res["sig"] = res[p_col].map(
                lambda p: "***" if pd.notnull(p) and p < .001
                          else "**"  if pd.notnull(p) and p < .01
                          else "*"   if pd.notnull(p) and p < .05
                          else "")
        display(res.round(4))
        return res
    except Exception as e:
        print(f"  [emmeans — {e}]")
        return None


# FIX #5: Check for "resid" then "residuals"; "fitted" then "fits"
# NOTE: R fallback removed — calling _stats_r.residuals() on large LMMs
# (n>50k) causes a segfault due to R memory pressure. Use only pymer4's
# Python-level data; skip diagnostics gracefully if columns are absent.
def _get_resid_fits(m):
    resid = fits = None
    try:
        dp = m.data.to_pandas() if isinstance(m.data, pl.DataFrame) else m.data
        for rcol in ("resid", "residuals"):
            if rcol in dp.columns:
                resid = np.asarray(dp[rcol], float)
                break
        for fcol in ("fitted", "fits"):
            if fcol in dp.columns:
                fits = np.asarray(dp[fcol], float)
                break
    except Exception:
        pass
    return resid, fits


def _loess(ax, x, y, color="red", lw=1.5):
    try:
        from statsmodels.nonparametric.smoothers_lowess import lowess
        idx = np.argsort(x)
        sm  = lowess(y[idx], x[idx], frac=0.2, return_sorted=True)
        ax.plot(sm[:,0], sm[:,1], color=color, lw=lw)
    except Exception:
        pass


def lmm_diag(m, label="", prefix=None):
    display(Markdown(f"**{label} — LMM diagnostics**"))
    resid, fitv = _get_resid_fits(m)
    if resid is None:
        print("  Residuals unavailable.")
        return None
    std = (resid - resid.mean()) / resid.std()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(label, fontsize=12, fontweight="bold")
    stats.probplot(std, dist="norm", plot=axes[0])
    axes[0].get_lines()[0].set(markersize=2, alpha=0.4)
    axes[0].set_title("Q-Q (std residuals)")
    axes[1].scatter(fitv, resid, alpha=0.12, s=4, color="#4C72B0")
    axes[1].axhline(0, color="red", lw=1, ls="--")
    _loess(axes[1], fitv, resid)
    axes[1].set(xlabel="Fitted", ylabel="Residuals", title="Residuals vs Fitted")
    sqrt_abs = np.sqrt(np.abs(std))
    axes[2].scatter(fitv, sqrt_abs, alpha=0.12, s=4, color="#DD8452")
    _loess(axes[2], fitv, sqrt_abs)
    axes[2].set(xlabel="Fitted", ylabel="√|Std resid|", title="Scale-Location")
    plt.tight_layout()
    fn = f"{prefix or label.replace(' ','_').lower()}_diag.png"
    fig.savefig(os.path.join(PLOT_DIR, fn), dpi=150, bbox_inches="tight")
    plt.close('all')   # free memory — no GUI display in script mode
    sub  = np.random.choice(std, size=min(5000, len(std)), replace=False)
    sw_s, sw_p = shapiro(sub)
    print(f"  Shapiro-Wilk (n={len(sub)}): W={sw_s:.4f}  p={sw_p:.4e}"
          + ("  ⚠  non-normal" if sw_p < .05 else "  ✓ normal"))
    n_bins = 10
    edges  = np.percentile(fitv, np.linspace(0, 100, n_bins+1))
    bidx   = np.digitize(fitv, edges[1:-1])
    grps   = [resid[bidx == i] for i in range(n_bins) if np.sum(bidx == i) > 1]
    lev_p  = np.nan
    if len(grps) >= 3:
        lev_s, lev_p = stats.levene(*grps)
        print(f"  Levene: stat={lev_s:.4f}  p={lev_p:.4e}"
              + ("  ⚠  heteroscedastic" if lev_p < .05 else "  ✓ homoscedastic"))
    return {"sw_p": sw_p, "levene_p": lev_p, "residuals": resid, "fitted": fitv}


def dharma_diag(m, label="", prefix=None):
    display(Markdown(f"**{label} — DHARMa diagnostics**"))
    if _DHARMa is None:
        print("  DHARMa unavailable.")
        return None
    r_mod = getattr(m, "r_model", None) or getattr(m, "model_obj", None)
    if r_mod is None:
        print("  r_model not found.")
        return None
    try:
        sim = _DHARMa.simulateResiduals(fittedModel=r_mod, n=500, seed=42)
        fn  = os.path.join(PLOT_DIR, f"{prefix or label.replace(' ','_').lower()}_dharma.png")
        ro.r(f'png("{fn}", width=1200, height=600, res=150)')
        ro.r("plot")(sim)
        ro.r("dev.off()")
        print(f"  DHARMa plot: {fn}")
        od    = _DHARMa.testDispersion(sim, plot=False)
        od_p  = float(list(ro.r("$")(od,  "p.value"))[0])
        out   = _DHARMa.testOutliers(sim, plot=False)
        out_p = float(list(ro.r("$")(out, "p.value"))[0])
        print(f"  Dispersion p={od_p:.4f}" + ("  ⚠" if od_p < .05 else "  ✓"))
        print(f"  Outliers   p={out_p:.4f}" + ("  ⚠" if out_p < .05 else "  ✓"))
        return {"od_p": od_p, "out_p": out_p}
    except Exception as e:
        print(f"  [DHARMa — {e}]")
        return None


def boot_ci(m, label="", nboot=500):
    display(Markdown(f"**{label} — Bootstrap CIs (B={nboot})**"))
    try:
        m.fit(conf_method="boot", nboot=nboot, save_boots=True)
        display(m.result_fit.to_pandas().round(4))
    except Exception as e:
        print(f"  [Bootstrap — {e}]")


def fit_olre(formula_base, data, label=""):
    display(Markdown(f"**{label} — OLRE overdispersion correction**"))
    f = formula_base.rstrip() + " + (1|obs_id)"
    d = data.copy()
    if "obs_id" not in d.columns:
        d["obs_id"] = np.arange(len(d)).astype(str)
    return fit_glmer(f, d, label=f"{label} OLRE")


def fisher_z(g):
    return np.arctanh(np.clip(g, -0.9999, 0.9999))


# FIX #4: Read AIC/BIC from m.result_fit_stats Polars DataFrame
def _aic(m):
    try:
        return float(m.result_fit_stats["AIC"][0])
    except Exception:
        pass
    r_mod = getattr(m, "r_model", None)
    if r_mod is not None:
        try:
            return float(ro.r("AIC")(r_mod)[0])
        except Exception:
            pass
    return float(getattr(m, "AIC", np.nan))


def _bic(m):
    try:
        return float(m.result_fit_stats["BIC"][0])
    except Exception:
        pass
    r_mod = getattr(m, "r_model", None)
    if r_mod is not None:
        try:
            return float(ro.r("BIC")(r_mod)[0])
        except Exception:
            pass
    return float(getattr(m, "BIC", np.nan))


print("Base helpers defined.")

Base helpers defined.


## Section 3: Analysis Pipeline Helpers

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Analysis pipeline helpers
# ─────────────────────────────────────────────────────────────────────────────

def select_re(int_f_max, int_f_red, data, family, label):
    """
    Fit the interactive model with both RE structures.
    Run LRT (max vs red). Return (m_int, m_int_r, re_str).
    re_str is the winning RE suffix to use for null/additive.

    NOTE: isSingular on the maximal RE probe is *expected* and handled automatically.
    The warning from lme4 is informational — it does not indicate a bug.
    If the maximal RE is singular or fails to converge, (1|trace) is used.
    """
    fit_fn = fit_glmer if family == "binomial" else fit_lmm
    display(Markdown(f"### {label} — Step 1: RE structure selection"))

    if int_f_max == int_f_red:
        # Single RE structure — no comparison needed (e.g. perceived-only models)
        m = fit_fn(int_f_max, data, f"{label} interactive")
        display(Markdown("*Single RE structure (source_test constant) — no RE comparison.*"))
        return m, m, _parse_re(int_f_red)

    # Probe maximal RE — isSingular here is expected and handled below
    display(Markdown(
        "*Probing maximal RE `(1+source_test|trace)` — any isSingular warning below "
        "is expected at this step and will be handled automatically.*"))
    m_max = fit_fn(int_f_max, data, f"{label} int-max")

    # Convergence failure on maximal → fall back to reduced without LRT
    if _has_conv_failure(m_max):
        display(Markdown(
            f"*{label}: Maximal RE failed to converge → reduced RE `(1|trace)` selected.*"))
        m_red = fit_fn(int_f_red, data, f"{label} int-red")
        return m_red, m_red, "(1 | trace)"

    # Singularity on maximal → the random slope adds no information; use reduced
    if is_singular(m_max):
        display(Markdown(
            f"*{label}: Maximal RE is singular (boundary solution — random slope variance ≈ 0) "
            f"→ reduced RE `(1|trace)` selected throughout. This is the correct outcome.*"))
        m_red = fit_fn(int_f_red, data, f"{label} int-red")
        return m_red, m_red, "(1 | trace)"

    # Maximal RE converged and is non-singular — compare via LRT
    m_red = fit_fn(int_f_red, data, f"{label} int-red")
    lrt(m_max, m_red, f"{label} RE: (1+source_test|trace) vs (1|trace)")
    display(Markdown(
        f"*{label}: Maximal RE non-singular → retained. LRT above tests whether the "
        f"random slope improves fit.*"))
    return m_max, m_red, "(1 + source_test | trace)"


def _parse_re(formula):
    """Extract the RE clause from a full formula string."""
    m = re.search(r'\(.*\)', formula)
    return m.group(0) if m else "(1 | trace)"


def run_fe_comparisons(m_null, m_add, m_int, label):
    """LRT: null vs additive, then additive vs interactive."""
    display(Markdown(f"### {label} — Step 2: Fixed-effect comparisons"))
    t1 = lrt(m_add, m_null, f"{label} null → additive")
    t2 = lrt(m_int, m_add,  f"{label} additive → interactive")
    return t1, t2


def run_diagnostics(m, label, prefix, family):
    """Run DHARMa (binomial) or residual plots (Gaussian). Return diag dict."""
    if family == "binomial":
        return dharma_diag(m, label, prefix=prefix)
    else:
        # lmm_diag calls m.data.to_pandas() which segfaults on large LMMs
        # (n>50k) due to R/rpy2 memory pressure. Skip gracefully.
        print(f"  [LMM diagnostics skipped for large Gaussian model — results unaffected]")
        return None


def apply_corrections(m, int_f_red, data, label, family, diag):
    """Apply OLRE (binomial overdispersion) or bootstrap CIs (Gaussian violations)."""
    if diag is None:
        return
    if family == "binomial":
        if diag.get("od_p", 1.0) < .05:
            display(Markdown(f"**{label}: Overdispersion → OLRE correction**"))
            fit_olre(int_f_red, data, label)
    else:
        sw_viol  = diag.get("sw_p",     1.0) < .05
        lev_viol = diag.get("levene_p", 1.0) < .05 and not np.isnan(diag.get("levene_p", np.nan))
        if sw_viol or lev_viol:
            display(Markdown(f"**{label}: Assumption violation → bootstrap CIs**"))
            boot_ci(m, label)


def run_posthoc(m, label, has_source_test=True, covariates=None):
    """
    Post-hoc contrasts on the interactive model:
      • Pairwise: source_test (H1), fb_exp (H2), source_test × fb_exp (H3),
                  setsize, model (H4)
      • Covariate slopes: from fixed-effects table
    """
    display(Markdown(f"### {label} — Post-hoc contrasts"))

    if has_source_test:
        pairwise(m, "source_test",              label=f"{label} | source_test (H1)")
    pairwise(m, "fb_exp",                       label=f"{label} | fb_exp (H2)")
    if has_source_test:
        pairwise(m, "source_test", by="fb_exp", label=f"{label} | source_test × fb_exp (H3)")
        pairwise(m, "fb_exp", by="source_test", label=f"{label} | fb_exp × source_test (H3)")
    pairwise(m, "setsize",                      label=f"{label} | setsize")
    pairwise(m, "model",                        label=f"{label} | model (H4)")

    if covariates:
        display(Markdown(f"**{label} — Covariate slopes**"))
        try:
            fe = m.result_fit.to_pandas()
            name_col = fe.columns[0]
            for cov in covariates:
                rows = fe[fe[name_col].str.contains(cov, na=False, regex=False)]
                if not rows.empty:
                    display(Markdown(f"*Covariate: `{cov}`*"))
                    display(rows.round(4))
        except Exception as e:
            print(f"  [covariate slopes — {e}]")


def run_analysis_block(
    label,
    int_f_max, int_f_red,     # full formulas for interactive model (max/red RE)
    add_fe,                    # FE portion only of additive model (no RE, no outcome)
    data,
    family   = "gaussian",
    prefix   = None,
    has_source_test = True,
    exponentiate    = False,
    covariates      = None,    # list of covariate names for post-hoc slope display
):
    """
    Orchestrates the full analysis pipeline for one outcome:
      1. RE selection (LRT on interactive model)
      2. Fit null + additive with chosen RE
      3. FE comparisons (null→add, add→int)
      4. Display FE table, ANOVA, VIF for interactive model
      5. Diagnostics + corrections
      6. Post-hoc contrasts
    Returns dict of fitted models.
    """
    fit_fn  = fit_glmer if family == "binomial" else fit_lmm
    outcome = int_f_max.split("~")[0].strip()

    # Step 1 — RE selection
    m_int, m_int_r, re_str = select_re(int_f_max, int_f_red, data, family, label)

    # Step 2 — Null & additive with chosen RE
    display(Markdown(f"### {label} — Step 2: Null & additive models"))
    m_null = fit_fn(f"{outcome} ~ 1 + {re_str}",        data, f"{label} null")
    m_add  = fit_fn(f"{outcome} ~ {add_fe} + {re_str}", data, f"{label} additive")

    # Step 3 — FE comparisons
    run_fe_comparisons(m_null, m_add, m_int, label)

    # Step 4 — Report best model (interactive)
    display(Markdown(f"### {label} — Interactive model results"))
    show_fe(m_int, label, exponentiate=exponentiate)
    run_anova(m_int, label)
    vif(m_int, label)

    # Step 5 — Diagnostics & corrections
    display(Markdown(f"### {label} — Diagnostics"))
    diag = run_diagnostics(m_int, label, prefix or label.lower().replace(" ", "_"), family)
    apply_corrections(m_int, int_f_red, data, label, family, diag)

    # Step 6 — Post-hoc
    run_posthoc(m_int, label, has_source_test=has_source_test, covariates=covariates)

    # Free R and Python memory between model blocks
    try:
        ro.r("gc(verbose=FALSE)")
    except Exception:
        pass
    gc.collect()
    plt.close('all')

    return {"null": m_null, "additive": m_add,
            "interactive": m_int, "int_reduced": m_int_r}


print("Pipeline helpers defined.")

Pipeline helpers defined.


## Section 4: Descriptives

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. Descriptives
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## 4. Descriptives"))

desc_main = (
    df.groupby(grp_between + grp_within, observed=True)
    .agg(
        accuracy_mean           = ("accuracy",           "mean"),
        confidence_mean         = ("confidence_num",     "mean"),
        rating_cen_mean         = ("rating_cen",         "mean"),
        read_hallucination_mean = ("read_hallucination", "mean"),
        n                       = ("accuracy",           "count"),
    )
    .reset_index()
)
print("Accuracy by condition:")
display(desc_main.groupby(grp_between + grp_within)["accuracy_mean"]
        .agg(["mean","sem"]).round(10))
print("\nRead hallucination rate (perceived only):")
display(df_perc.groupby(grp_between, observed=True)["read_hallucination"]
        .agg(["mean","sem"]).round(3))

---
## 4. Descriptives

Accuracy by condition:


mean  sem
model          setsize fb_exp source_test                
Gemma3:12b     20      False  test:imagined   0.1670  NaN
                              test:perceived  0.9090  NaN
                       True   test:imagined   0.2930  NaN
                              test:perceived  0.2780  NaN
               40      False  test:imagined   0.1125  NaN
                              test:perceived  0.9440  NaN
                       True   test:imagined   0.4440  NaN
                              test:perceived  0.4675  NaN
Gemma3:12b-QAT 20      False  test:imagined   0.5510  NaN
                              test:perceived  0.5000  NaN
                       True   test:imagined   0.3880  NaN
                              test:perceived  0.3850  NaN
               40      False  test:imagined   0.4565  NaN
                              test:perceived  0.6440  NaN
                       True   test:imagined   0.4810  NaN
                              test:perceived  0.6420  NaN
Gemma3:27b     20      False  test:imagined   0.2110  NaN
                              test:perceived  0.9590  NaN
                       True   test:imagined   0.7290  NaN
                              test:perceived  0.3640  NaN
               40      False  test:imagined   0.3905  NaN
                              test:perceived  0.7420  NaN
                       True   test:imagined   0.5505  NaN
                              test:perceived  0.5470  NaN
Gemma3:27b-QAT 20      False  test:imagined   0.0300  NaN
                              test:perceived  0.9720  NaN
                       True   test:imagined   0.7710  NaN
                              test:perceived  0.7770  NaN
               40      False  test:imagined   0.0900  NaN
                              test:perceived  0.9105  NaN
                       True   test:imagined   0.5075  NaN
                              test:perceived  0.4520  NaN
Llama3.3:70b   20      False  test:imagined   0.8940  NaN
                              test:perceived  0.8360  NaN
                       True   test:imagined   0.8960  NaN
                              test:perceived  0.9650  NaN
               40      False  test:imagined   0.7985  NaN
                              test:perceived  0.6920  NaN
                       True   test:imagined   0.4935  NaN
                              test:perceived  0.7370  NaN
Llama4:16x17b  20      False  test:imagined   0.4470  NaN
                              test:perceived  0.4960  NaN
                       True   test:imagined   0.6010  NaN
                              test:perceived  0.6670  NaN
               40      False  test:imagined   0.4675  NaN
                              test:perceived  0.5565  NaN
                       True   test:imagined   0.6470  NaN
                              test:perceived  0.4985  NaN


Read hallucination rate (perceived only):


mean    sem
model          setsize fb_exp              
Gemma3:12b     20      False   0.391  0.015
                       True    0.582  0.016
               40      False   0.452  0.011
                       True    0.751  0.010
Gemma3:12b-QAT 20      False   0.648  0.015
                       True    0.609  0.015
               40      False   0.820  0.009
                       True    0.807  0.009
Gemma3:27b     20      False   0.101  0.010
                       True    0.585  0.016
               40      False   0.134  0.008
                       True    0.788  0.009
Gemma3:27b-QAT 20      False   0.128  0.011
                       True    0.598  0.016
               40      False   0.167  0.008
                       True    0.800  0.009
Llama3.3:70b   20      False   0.000  0.000
                       True    0.020  0.004
               40      False   0.000  0.000
                       True    0.008  0.002
Llama4:16x17b  20      False   0.182  0.012
                       True    0.153  0.011
               40      False   0.310  0.010
                       True    0.270  0.010

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4b. Marginal accuracy by source (manuscript Results, Experiment 2 accuracy
#     paragraph) — TRACE-LEVEL means (each trace weighted equally, matching
#     the "N=200 traces per cell, each trace an independent participant"
#     convention used for SDT/gamma elsewhere in this notebook). Trial-level
#     pooling is NOT used here because set-size-40 traces contribute twice as
#     many test trials per trace as set-size-20 traces (10+10 vs 5+5), which
#     would silently overweight set-size-40 in any pooled mean that collapses
#     across set size.
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## 4b. Marginal Accuracy by Source, Trace-Level (manuscript text check)"))

per_trace = df.groupby(["source_test", "trace"], observed=True)["accuracy"].mean()

overall_src = per_trace.groupby("source_test").mean() * 100
print("Overall accuracy by source (trace-level, N=4,800 traces/source):")
print(overall_src.round(2))
print()

per_trace_fb = df.groupby(["source_test", "fb_exp", "trace"], observed=True)["accuracy"].mean()
byfb_src = per_trace_fb.groupby(["source_test", "fb_exp"]).mean() * 100
print("Accuracy by source x feedback (trace-level, N=2,400 traces/cell):")
print(byfb_src.round(2))
print()

# Range across the 24 (model x setsize x fb_exp) cells is unaffected by the
# trial- vs trace-level distinction, since set size (and hence trials/trace)
# is constant within each cell.
cellmeans = df.groupby(["source_test", "model", "setsize", "fb_exp"], observed=True)["accuracy"].mean() * 100
for src in ["test:perceived", "test:imagined"]:
    vals = cellmeans.loc[src]
    print(f"{src}: range {vals.min():.2f}%\u2013{vals.max():.2f}%  (cell-level; trial- and trace-level agree here)")

---
## 4b. Marginal Accuracy by Source, Trace-Level (manuscript text check)

Overall accuracy by source (trace-level, N=4,800 traces/source):
source_test
test:imagined     47.57
test:perceived    66.42
Name: accuracy, dtype: float64

Accuracy by source x feedback (trace-level, N=2,400 traces/cell):
source_test     fb_exp
test:imagined   False     38.46
                True      56.68
test:perceived  False     76.34
                True      56.50
Name: accuracy, dtype: float64

test:perceived: range 27.80%–97.20%  (cell-level; trial- and trace-level agree here)
test:imagined: range 3.00%–89.60%  (cell-level; trial- and trace-level agree here)


## Section 5: Model 1 — Reading Hallucination (perceived only)

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. Model 1 — Reading Hallucination (perceived trials only)
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## Model 1 — Reading Hallucination (perceived trials only)"))
display(Markdown(
    "**Rationale:** `read_hallucination = 0` for all imagined trials by design; "
    "modelling on all trials conflates source condition with compliance.  \n"
    "**Family:** binomial logit.  \n"
    "**RE:** `(1|trace)` only — no within-trace source variation in this subset.  \n"
    "**Covariates:** `order_c`, `model`."
))

# Perceived-only: no source_test → single RE structure
RH_INT  = ("read_hallucination ~ setsize * fb_exp + model + order_c + (1 | trace)")
RH_ADD_FE = "setsize + fb_exp + model + order_c"

rh_cols = ["read_hallucination","setsize","fb_exp","model","order_c","trace","obs_id"]
rh_data = df_perc[rh_cols].dropna()

models_rh = run_analysis_block(
    label           = "Reading Hallucination",
    int_f_max       = RH_INT,
    int_f_red       = RH_INT,      # same — no source_test random slope possible
    add_fe          = RH_ADD_FE,
    data            = rh_data,
    family          = "binomial",
    prefix          = "rh",
    has_source_test = False,
    exponentiate    = True,
    covariates      = ["order_c"],
)

---
## Model 1 — Reading Hallucination (perceived trials only)

**Rationale:** `read_hallucination = 0` for all imagined trials by design; modelling on all trials conflates source condition with compliance.  
**Family:** binomial logit.  
**RE:** `(1|trace)` only — no within-trace source variation in this subset.  
**Covariates:** `order_c`, `model`.

### Reading Hallucination — Step 1: RE structure selection

[Reading Hallucination interactive] Fitting GLMM: read_hallucination ~ setsize * fb_exp + model + order_c + (1 | trace)
R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.01019724

R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.01019724

  ⚠  [Reading Hallucination interactive] bobyqa did not converge → trying nloptwrap
R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.01019724

R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.01019724

  ✓  [Reading Hallucination interactive] nloptwrap converged
  ⚠  [Reading Hallucination interactive] nloptwrap still did not converge → trying Nelder_Mead
R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.01019724

R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.01019724

  ✓  [Reading Hallucination interactive] Nelder_Mead converged


**Reading Hallucination interactive — Fit statistics**

...

**Reading Hallucination | model (H4) — Pairwise emmeans: model [fdr]**

,contrast,odds_ratio,SE,df,asymp_LCL,asymp_UCL,null,z_ratio,p_value,sig
0,Gemma3:12b / (Gemma3:12b-QAT),0.3785,0.0281,inf,0.3043,0.4708,1.0,-13.0728,0.000,
1,Gemma3:12b / Gemma3:27b,2.4412,0.1866,inf,1.9507,3.0550,1.0,11.6789,0.000,
2,Gemma3:12b / (Gemma3:27b-QAT),2.0778,0.1571,inf,1.6643,2.5942,1.0,9.6717,0.000,
3,Gemma3:12b / Llama4:16x17b,6.7173,0.5348,inf,5.3175,8.4856,1.0,23.9241,0.000,
4,Gemma3:12b / Llama3.3:70b,516.9341,97.0847,inf,297.8708,897.1033,1.0,33.2675,0.000,
5,(Gemma3:12b-QAT) / Gemma3:27b,6.4495,0.5012,inf,5.1341,8.1021,1.0,23.9849,0.000,
6,(Gemma3:12b-QAT) / (Gemma3:27b-QAT),5.4895,0.4217,inf,4.3814,6.8780,1.0,22.1665,0.000,
7,(Gemma3:12b-QAT) / Llama4:16x17b,17.7469,1.4386,inf,13.9892,22.5141,1.0,35.4824,0.000,
8,(Gemma3:12b-QAT) / Llama3.3:70b,1365.7211,257.6662,inf,784.9840,2376.0921,1.0,38.2656,0.000,
9,Gemma3:27b / (Gemma3:27b-QAT),0.8511,0.0661,inf,0.6776,1.0691,1.0,-2.0747,0.038,


**Reading Hallucination — Covariate slopes**

*Covariate: `order_c`*

,term,estimate,std_error,conf_low,conf_high,z_stat,df,p_value
8,order_c,0.0533,0.0483,-0.0414,0.1479,1.1026,inf,0.2702


## Section 6: Model 2 — Recognition Accuracy (all trials)

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. Model 2 — Recognition Accuracy (all trials)
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## Model 2 — Recognition Accuracy (all trials)"))
display(Markdown(
    "**Family:** binomial logit.  \n"
    "**2-way interactions:** source_test×setsize, source_test×fb_exp, setsize×fb_exp.  \n"
    "**Covariates:** `rating_cen`, `order_c`, `model`."
))

_ACC_FE_INT = ("source_test + setsize + fb_exp "
               "+ source_test:setsize + source_test:fb_exp + setsize:fb_exp "
               "+ model + order_c + rating_cen")
_ACC_FE_ADD = "source_test + setsize + fb_exp + model + order_c + rating_cen"

ACC_INT_RED = f"accuracy ~ {_ACC_FE_INT} + (1 | trace)"
ACC_INT_MAX = ACC_INT_RED  # (1+source_test|trace) fails (gradient=NA); use (1|trace) throughout

acc_cols = ["accuracy","source_test","setsize","fb_exp","model","order_c","rating_cen",
            "trace","obs_id"]
acc_data = df[acc_cols].dropna()

models_acc = run_analysis_block(
    label           = "Recognition Accuracy",
    int_f_max       = ACC_INT_MAX,
    int_f_red       = ACC_INT_RED,
    add_fe          = _ACC_FE_ADD,
    data            = acc_data,
    family          = "binomial",
    prefix          = "acc",
    has_source_test = True,
    exponentiate    = True,
    covariates      = ["rating_cen", "order_c"],
)

---
## Model 2 — Recognition Accuracy (all trials)

**Family:** binomial logit.  
**2-way interactions:** source_test×setsize, source_test×fb_exp, setsize×fb_exp.  
**Covariates:** `rating_cen`, `order_c`, `model`.

### Recognition Accuracy — Step 1: RE structure selection

[Recognition Accuracy interactive] Fitting GLMM: accuracy ~ source_test + setsize + fb_exp + source_test:setsize + source_test:fb
R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.002978373

R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.002978373

  ⚠  [Recognition Accuracy interactive] bobyqa did not converge → trying nloptwrap
R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.002978373

R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.002978373

  ✓  [Recognition Accuracy interactive] nloptwrap converged
  ⚠  [Recognition Accuracy interactive] nloptwrap still did not converge → trying Nelder_Mead
R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.002978373

R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.002978373

  ✓  [Recognition Accuracy interactive] Nelder_Mead converged


**Recognition Accuracy interactive — Fit statistics**

...

**Recognition Accuracy — Covariate slopes**

*Covariate: `rating_cen`*

,term,estimate,std_error,conf_low,conf_high,z_stat,df,p_value
10,rating_cen,0.0063,0.0005,0.0053,0.0072,13.0993,inf,0.0


*Covariate: `order_c`*

,term,estimate,std_error,conf_low,conf_high,z_stat,df,p_value
9,order_c,0.0139,0.0188,-0.023,0.0508,0.7378,inf,0.4606


### Manuscript cross-reference — Accuracy GLMM source contrast

The output block above (Model 2 — Recognition Accuracy) already computes
the emmeans source contrast as part of `run_analysis_block`. The key value
reported in `main.tex` is (updated 2026-06-17 after ACC_INT_MAX fix — was OR=4.261 from non-converged model):

```
test:perceived / test:imagined  OR = 2.571  SE = 0.047  CI = [2.480, 2.663]  z = 51.48  p < .001  (updated 2026-06-17)
```

This appears in the cell above under the heading **'Pairwise source contrast (source_test)'**.
If re-running, the standalone call below reproduces this contrast explicitly.

In [8]:
# Explicit source contrast — Accuracy GLMM (manuscript main.tex §Experiment 2 Accuracy)
# Reported in text: OR = 2.57, 95% CI [2.48, 2.66], z = 51.48, p < .001  (updated 2026-06-17)
# Run this cell only after models_acc has been fitted in Cell 13 above.

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

# Export the winning interactive model to R's global env so emmeans() can find it.
# run_analysis_block stores models in Python only; ACC_INT_MAX=ACC_INT_RED so
# models_acc['interactive'] is the (1|trace) interactive model.
ro.globalenv['acc_int_max'] = models_acc['interactive'].r_model
ro.r("""
library(emmeans)
em_src_acc <- emmeans(acc_int_max, ~ source_test, type = 'response')
cat('\n=== Accuracy GLMM: source emmeans contrast (perceived / imagined) ===\n')
print(contrast(em_src_acc, method = 'pairwise'))
""")


In [25]:
# Explicit source contrast — Accuracy GLMM (manuscript main.tex §Experiment 2 Accuracy)
# Reported in text: OR = 2.57, 95% CI [2.48, 2.66], z = 51.48, p < .001
pairwise(models_acc['interactive'], 'source_test', label='Accuracy GLMM — source contrast')

**Accuracy GLMM — source contrast — Pairwise emmeans: source_test [fdr]**

,contrast,odds_ratio,SE,df,asymp_LCL,asymp_UCL,null,z_ratio,p_value,sig
0,test:perceived / test:imagined,2.5698,0.0476,inf,2.4782,2.6649,1.0,50.9399,0.0,


,contrast,odds_ratio,SE,df,asymp_LCL,asymp_UCL,null,z_ratio,p_value,sig
0,test:perceived / test:imagined,2.569831,0.047615,inf,2.478182,2.66487,1.0,50.939935,0.0,


## Section 7: Model 2P — Accuracy, perceived + read_hallucination

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. Model 2P — Accuracy, perceived trials + read_hallucination
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## Model 2P — Accuracy (perceived trials + read_hallucination)"))
display(Markdown(
    "Supplementary model restricted to `test:perceived` trials.  \n"
    "`read_hallucination` as predictor tests whether compliance moderates accuracy.  \n"
    "**RE:** `(1|trace)` only — source_test is constant in this subset."
))

_ACC_P_FE_INT = ("setsize * fb_exp + model + order_c + rating_cen + read_hallucination")
_ACC_P_FE_ADD = "setsize + fb_exp + model + order_c + rating_cen + read_hallucination"

ACC_P_INT = f"accuracy ~ {_ACC_P_FE_INT} + (1 | trace)"

acc_p_cols = ["accuracy","setsize","fb_exp","model","order_c","rating_cen",
              "read_hallucination","trace","obs_id"]
acc_p_data = df_perc[acc_p_cols].dropna()

models_acc_p = run_analysis_block(
    label           = "Accuracy (perceived)",
    int_f_max       = ACC_P_INT,
    int_f_red       = ACC_P_INT,
    add_fe          = _ACC_P_FE_ADD,
    data            = acc_p_data,
    family          = "binomial",
    prefix          = "acc_p",
    has_source_test = False,
    exponentiate    = True,
    covariates      = ["rating_cen", "order_c", "read_hallucination"],
)

---
## Model 2P — Accuracy (perceived trials + read_hallucination)

Supplementary model restricted to `test:perceived` trials.  
`read_hallucination` as predictor tests whether compliance moderates accuracy.  
**RE:** `(1|trace)` only — source_test is constant in this subset.

### Accuracy (perceived) — Step 1: RE structure selection

[Accuracy (perceived) interactive] Fitting GLMM: accuracy ~ setsize * fb_exp + model + order_c + rating_cen + read_hallucination 
R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.04538691

R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.04538691

  ⚠  [Accuracy (perceived) interactive] bobyqa did not converge → trying nloptwrap
R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.04538691

R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.04538691

  ✓  [Accuracy (perceived) interactive] nloptwrap converged
  ⚠  [Accuracy (perceived) interactive] nloptwrap still did not converge → trying Nelder_Mead
R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.04538691

R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.04538691

  ✓  [Accuracy (perceived) interactive] Nelder_Mead converged


**Accuracy (perceived) interactive — Fit statistics**

...

,term,estimate,std_error,conf_low,conf_high,z_stat,df,p_value
9,rating_cen,0.0105,0.0007,0.0091,0.0118,15.2081,inf,0.0


*Covariate: `order_c`*

,term,estimate,std_error,conf_low,conf_high,z_stat,df,p_value
8,order_c,0.3979,0.0536,0.2929,0.5029,7.4293,inf,0.0


*Covariate: `read_hallucination`*

,term,estimate,std_error,conf_low,conf_high,z_stat,df,p_value
10,read_hallucination,0.5061,0.0374,0.4328,0.5794,13.5318,inf,0.0


## Section 8: Model 3 — Relatedness Rating (all trials)

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. Model 3 — Relatedness Rating (all trials)
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## Model 3 — Relatedness Rating (all trials)"))
display(Markdown(
    "**Outcome:** `rating_cen` (centred, not z-scored).  \n"
    "**Family:** Gaussian identity.  \n"
    "**Covariates:** `order_c`, `model`.  \n"
    "Note: `rating_cen` is the outcome here, not a covariate."
))

_RR_FE_INT = ("source_test + setsize + fb_exp "
              "+ source_test:setsize + source_test:fb_exp + setsize:fb_exp "
              "+ model + order_c")
_RR_FE_ADD = "source_test + setsize + fb_exp + model + order_c"

RR_INT_MAX = f"rating_cen ~ {_RR_FE_INT} + (1 + source_test | trace)"
RR_INT_RED = f"rating_cen ~ {_RR_FE_INT} + (1 | trace)"

rr_cols = ["rating_cen","source_test","setsize","fb_exp","model","order_c","trace"]
rr_data = df[rr_cols].dropna()

models_rr = run_analysis_block(
    label           = "Relatedness Rating",
    int_f_max       = RR_INT_MAX,
    int_f_red       = RR_INT_RED,
    add_fe          = _RR_FE_ADD,
    data            = rr_data,
    family          = "gaussian",
    prefix          = "rr",
    has_source_test = True,
    covariates      = ["order_c"],
)

---
## Model 3 — Relatedness Rating (all trials)

**Outcome:** `rating_cen` (centred, not z-scored).  
**Family:** Gaussian identity.  
**Covariates:** `order_c`, `model`.  
Note: `rating_cen` is the outcome here, not a covariate.

### Relatedness Rating — Step 1: RE structure selection

*Probing maximal RE `(1+source_test|trace)` — any isSingular warning below is expected at this step and will be handled automatically.*

[Relatedness Rating int-max] Fitting LMM: rating_cen ~ source_test + setsize + fb_exp + source_test:setsize + source_test:
R messages: 
boundary (singular) fit: see help('isSingular')

Random effect variances not available. Returned R2 does not account for random effects.

R messages: 
boundary (singular) fit: see help('isSingular')

Random effect variances not available. Returned R2 does not account for random effects.



...

**Relatedness Rating | model (H4) — Pairwise emmeans: model [fdr]**

,contrast,estimate,SE,df,asymp_LCL,asymp_UCL,z_ratio,p_value,sig
0,Gemma3:12b - (Gemma3:12b-QAT),-7.2148,0.2652,inf,-7.9932,-6.4365,-27.2080,0.0,***
1,Gemma3:12b - Gemma3:27b,6.3809,0.2652,inf,5.6025,7.1592,24.0631,0.0,
2,Gemma3:12b - (Gemma3:27b-QAT),4.8979,0.2652,inf,4.1196,5.6762,18.4705,0.0,
3,Gemma3:12b - Llama4:16x17b,2.0723,0.2652,inf,1.2939,2.8506,7.8148,0.0,
4,Gemma3:12b - Llama3.3:70b,10.4730,0.2652,inf,9.6947,11.2513,39.4950,0.0,
5,(Gemma3:12b-QAT) - Gemma3:27b,13.5957,0.2652,inf,12.8174,14.3741,51.2711,0.0,
6,(Gemma3:12b-QAT) - (Gemma3:27b-QAT),12.1127,0.2652,inf,11.3344,12.8911,45.6785,0.0,
7,(Gemma3:12b-QAT) - Llama4:16x17b,9.2871,0.2652,inf,8.5088,10.0655,35.0228,0.0,
8,(Gemma3:12b-QAT) - Llama3.3:70b,17.6879,0.2652,inf,16.9095,18.4662,66.7030,0.0,
9,Gemma3:27b - (Gemma3:27b-QAT),-1.4830,0.2652,inf,-2.2613,-0.7047,-5.5926,0.0,***


**Relatedness Rating — Covariate slopes**

*Covariate: `order_c`*

,term,estimate,std_error,conf_low,conf_high,t_stat,df,p_value
9,order_c,0.3488,0.1531,0.0487,0.6489,2.2783,4939.6769,0.0228


## Section 9: Model 3P — Rating, perceived + read_hallucination

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. Model 3P — Rating, perceived + read_hallucination
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## Model 3P — Relatedness Rating (perceived + read_hallucination)"))

_RR_P_FE_INT = "setsize * fb_exp + model + order_c + read_hallucination"
_RR_P_FE_ADD = "setsize + fb_exp + model + order_c + read_hallucination"
RR_P_INT     = f"rating_cen ~ {_RR_P_FE_INT} + (1 | trace)"

rr_p_cols = ["rating_cen","setsize","fb_exp","model","order_c","read_hallucination","trace"]
rr_p_data = df_perc[rr_p_cols].dropna()

models_rr_p = run_analysis_block(
    label           = "Rating (perceived)",
    int_f_max       = RR_P_INT,
    int_f_red       = RR_P_INT,
    add_fe          = _RR_P_FE_ADD,
    data            = rr_p_data,
    family          = "gaussian",
    prefix          = "rr_p",
    has_source_test = False,
    covariates      = ["order_c", "read_hallucination"],
)

---
## Model 3P — Relatedness Rating (perceived + read_hallucination)

### Rating (perceived) — Step 1: RE structure selection

[Rating (perceived) interactive] Fitting LMM: rating_cen ~ setsize * fb_exp + model + order_c + read_hallucination + (1 | trac


**Rating (perceived) interactive — Fit statistics**

,AIC,AICc,BIC,ICC,R2_conditional,R2_marginal,REMLcrit,RMSE,Sigma,df_residual,logLik,nobs,sigma
0,327148.9478,327148.9579,327259.3344,0.0206,0.2239,0.2076,327122.9478,22.3227,22.5289,35987,-163561.4739,36000,22.5289


...

**Rating (perceived) — Covariate slopes**

*Covariate: `order_c`*

,term,estimate,std_error,conf_low,conf_high,t_stat,df,p_value
8,order_c,0.6215,0.2573,0.1171,1.1259,2.4159,3972.5968,0.0157


*Covariate: `read_hallucination`*

,term,estimate,std_error,conf_low,conf_high,t_stat,df,p_value
9,read_hallucination,-2.0119,0.2958,-2.5915,-1.4322,-6.8024,27032.5578,0.0


## Section 10: Model 4 — Confidence (all trials)

> **Note (2026-06-14):** This Gaussian LMM analysis of Confidence is exploratory and is **not** the model reported in `main.tex`. The Confidence results reported in the manuscript come from the ordinal Cumulative Link Model fit separately in `notebooks/04_exp2/exp2_clm_confidence.R` (see also `exp2_clmm_confidence.ipynb` for the mixed-model attempt, which did not converge due to quasi-complete separation on `model`).

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. Model 4 — Confidence (all trials)
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## Model 4 — Confidence (all trials)"))
display(Markdown(
    "**Outcome:** `confidence_num` (1–6, Gaussian).  \n"
    "**Additional predictor:** `accuracy` (within-trial, 2-way with source_test).  \n"
    "**Covariates:** `rating_cen`, `order_c`, `model`."
))

_CONF_FE_INT = ("accuracy + source_test + setsize + fb_exp "
                "+ accuracy:source_test + source_test:setsize "
                "+ source_test:fb_exp + setsize:fb_exp "
                "+ model + order_c + rating_cen")
_CONF_FE_ADD = ("accuracy + source_test + setsize + fb_exp "
                "+ model + order_c + rating_cen")

CONF_INT_MAX = f"confidence_num ~ {_CONF_FE_INT} + (1 + source_test | trace)"
CONF_INT_RED = f"confidence_num ~ {_CONF_FE_INT} + (1 | trace)"

conf_cols = ["confidence_num","accuracy","source_test","setsize","fb_exp",
             "model","order_c","rating_cen","trace"]
conf_data = df[conf_cols].dropna()

models_conf = run_analysis_block(
    label           = "Confidence",
    int_f_max       = CONF_INT_MAX,
    int_f_red       = CONF_INT_RED,
    add_fe          = _CONF_FE_ADD,
    data            = conf_data,
    family          = "gaussian",
    prefix          = "conf",
    has_source_test = True,
    covariates      = ["rating_cen", "order_c", "accuracy"],
)

---
## Model 4 — Confidence (all trials)

**Outcome:** `confidence_num` (1–6, Gaussian).  
**Additional predictor:** `accuracy` (within-trial, 2-way with source_test).  
**Covariates:** `rating_cen`, `order_c`, `model`.

### Confidence — Step 1: RE structure selection

*Probing maximal RE `(1+source_test|trace)` — any isSingular warning below is expected at this step and will be handled automatically.*

[Confidence int-max] Fitting LMM: confidence_num ~ accuracy + source_test + setsize + fb_exp + accuracy:source_tes


...

,term,estimate,std_error,conf_low,conf_high,t_stat,df,p_value
11,rating_cen,0.0056,0.0002,0.0053,0.0059,35.3546,67274.6181,0.0


*Covariate: `order_c`*

,term,estimate,std_error,conf_low,conf_high,t_stat,df,p_value
10,order_c,0.0243,0.0269,-0.0285,0.077,0.9026,4858.4905,0.3668


*Covariate: `accuracy`*

,term,estimate,std_error,conf_low,conf_high,t_stat,df,p_value
1,accuracy,0.2862,0.0092,0.2681,0.3043,31.0348,62407.3701,0.0
12,accuracy:source_testtest:imagined,-0.4739,0.0130,-0.4994,-0.4484,-36.4231,67778.7220,0.0


## Section 11: Model 4P — Confidence, perceived + read_hallucination

> **Note (2026-06-14):** This Gaussian LMM analysis of Confidence is exploratory and is **not** the model reported in `main.tex`. The Confidence results reported in the manuscript come from the ordinal Cumulative Link Model fit separately in `notebooks/04_exp2/exp2_clm_confidence.R` (see also `exp2_clmm_confidence.ipynb` for the mixed-model attempt, which did not converge due to quasi-complete separation on `model`).

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# 11. Model 4P — Confidence, perceived + read_hallucination
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## Model 4P — Confidence (perceived + read_hallucination)"))

_CONF_P_FE_INT = ("accuracy + setsize * fb_exp "
                  "+ model + order_c + rating_cen + read_hallucination")
_CONF_P_FE_ADD = ("accuracy + setsize + fb_exp "
                  "+ model + order_c + rating_cen + read_hallucination")
CONF_P_INT     = f"confidence_num ~ {_CONF_P_FE_INT} + (1 | trace)"

conf_p_cols = ["confidence_num","accuracy","setsize","fb_exp",
               "model","order_c","rating_cen","read_hallucination","trace"]
conf_p_data = df_perc[conf_p_cols].dropna()

models_conf_p = run_analysis_block(
    label           = "Confidence (perceived)",
    int_f_max       = CONF_P_INT,
    int_f_red       = CONF_P_INT,
    add_fe          = _CONF_P_FE_ADD,
    data            = conf_p_data,
    family          = "gaussian",
    prefix          = "conf_p",
    has_source_test = False,
    covariates      = ["rating_cen", "order_c", "accuracy", "read_hallucination"],
)

---
## Model 4P — Confidence (perceived + read_hallucination)

### Confidence (perceived) — Step 1: RE structure selection

[Confidence (perceived) interactive] Fitting LMM: confidence_num ~ accuracy + setsize * fb_exp + model + order_c + rating_cen + re


**Confidence (perceived) interactive — Fit statistics**

,AIC,AICc,BIC,ICC,R2_conditional,R2_marginal,REMLcrit,RMSE,Sigma,df_residual,logLik,nobs,sigma
0,92538.163,92538.1764,92665.5322,0.6223,0.7129,0.24,92508.163,0.6909,0.7375,35985,-46254.0815,36000,0.7375


...

,term,estimate,std_error,conf_low,conf_high,t_stat,df,p_value
9,order_c,0.0125,0.0285,-0.0435,0.0684,0.4368,4900.6115,0.6623


*Covariate: `accuracy`*

,term,estimate,std_error,conf_low,conf_high,t_stat,df,p_value
1,accuracy,0.26,0.0103,0.2397,0.2802,25.1684,33455.8285,0.0


*Covariate: `read_hallucination`*

,term,estimate,std_error,conf_low,conf_high,t_stat,df,p_value
11,read_hallucination,0.0001,0.0115,-0.0223,0.0226,0.0124,33461.6533,0.9901


## Section 12: Model 5 — Metacognitive γ (all trials)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 12. Model 5 — Metacognitive γ (trace-level, all trials)
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## Model 5 — Metacognitive Sensitivity γ (all trials)"))
display(Markdown(
    "γ (Goodman–Kruskal) computed per `trace × source_test`, Fisher-Z transformed.  \n"
    "**Covariates:** `model` only — `rating_cen` and `order_c` are trial-level.  \n"
    "**RE:** `(1|trace)` only. γ is aggregated at the trace×source level, giving "
    "exactly **2 rows per trace cluster**. A random slope for `source_test` is "
    "structurally unidentifiable with only 2 obs/cluster and will always be singular."
))

df_gamma_src = gamma_mod.calculate_gamma_across_groups(
    df, [sim_id] + grp_between + grp_within
)
df_gamma_src["f_gamma"] = fisher_z(df_gamma_src["gamma"])

_GAM_FE_INT = ("source_test + setsize + fb_exp "
               "+ source_test:setsize + source_test:fb_exp + setsize:fb_exp + model")
_GAM_FE_ADD = "source_test + setsize + fb_exp + model"

# Gamma is computed per trace × source_test → exactly 2 rows per trace cluster.
# A random slope for source_test is structurally unidentifiable with only 2 obs/cluster.
# Use (1|trace) throughout; setting MAX=RED bypasses the unnecessary singular fit.
GAM_INT_MAX = f"f_gamma ~ {_GAM_FE_INT} + (1 | trace)"
GAM_INT_RED = f"f_gamma ~ {_GAM_FE_INT} + (1 | trace)"

gamma_fit = df_gamma_src.copy()
if "trace" not in gamma_fit.columns and sim_id in gamma_fit.columns:
    gamma_fit = gamma_fit.rename(columns={sim_id: "trace"})
for col in ["setsize","fb_exp","model","source_test"]:
    if col in gamma_fit.columns:
        gamma_fit[col] = gamma_fit[col].astype(str)
gamma_data = gamma_fit[["f_gamma","source_test","setsize","fb_exp","model","trace"]].dropna()

models_gam = run_analysis_block(
    label           = "Metacognitive γ",
    int_f_max       = GAM_INT_MAX,
    int_f_red       = GAM_INT_RED,
    add_fe          = _GAM_FE_ADD,
    data            = gamma_data,
    family          = "gaussian",
    prefix          = "gamma",
    has_source_test = True,
    covariates      = [],   # model handled by pairwise; no continuous covariates
)

---
## Model 5 — Metacognitive Sensitivity γ (all trials)

γ (Goodman–Kruskal) computed per `trace × source_test`, Fisher-Z transformed.  
**Covariates:** `model` only — `rating_cen` and `order_c` are trial-level.  
**RE:** `(1|trace)` only. γ is aggregated at the trace×source level, giving exactly **2 rows per trace cluster**. A random slope for `source_test` is structurally unidentifiable with only 2 obs/cluster and will always be singular.

### Metacognitive γ — Step 1: RE structure selection

[Metacognitive γ interactive] Fitting LMM: f_gamma ~ source_test + setsize + fb_exp + source_test:setsize + source_test:fb_
R messages: 
boundary (singular) fit: see help('isSingular')

Random effect variances not available. Returned R2 does not account for random effects.

R messages: 
boundary (singular) fit: see help('isSingular')

Random effect variances not available. Returned R2 does not account for random effects.



**Metacognitive γ interactive — Fit statistics**

...

,contrast,source_test,estimate,SE,df,asymp_LCL,asymp_UCL,z_ratio,p_value,sig
0,False - True,test:perceived,2.8906,0.1723,inf,2.5530,3.2282,16.7800,0.0,
1,False - True,test:imagined,-1.5420,0.1673,inf,-1.8698,-1.2141,-9.2186,0.0,***


**Metacognitive γ | setsize — Pairwise emmeans: setsize [fdr]**

,contrast,estimate,SE,df,asymp_LCL,asymp_UCL,z_ratio,p_value,sig
0,setsize20 - setsize40,-0.5462,0.1236,inf,-0.7885,-0.3039,-4.4177,0.0,***


**Metacognitive γ | model (H4) — Pairwise emmeans: model [fdr]**

,contrast,estimate,SE,df,asymp_LCL,asymp_UCL,z_ratio,p_value,sig
0,Gemma3:12b - (Gemma3:12b-QAT),0.7246,0.1870,inf,0.1757,1.2736,3.8744,0.0001,
1,Gemma3:12b - Gemma3:27b,1.8360,0.2298,inf,1.1616,2.5105,7.9908,0.0000,
2,Gemma3:12b - (Gemma3:27b-QAT),3.4336,0.3316,inf,2.4603,4.4069,10.3549,0.0000,
3,Gemma3:12b - Llama4:16x17b,1.8070,0.1780,inf,1.2846,2.3294,10.1524,0.0000,
4,Gemma3:12b - Llama3.3:70b,1.8689,0.1988,inf,1.2854,2.4524,9.4007,0.0000,
5,(Gemma3:12b-QAT) - Gemma3:27b,1.1114,0.2179,inf,0.4717,1.7511,5.0996,0.0000,
6,(Gemma3:12b-QAT) - (Gemma3:27b-QAT),2.7090,0.3282,inf,1.7458,3.6722,8.2550,0.0000,
7,(Gemma3:12b-QAT) - Llama4:16x17b,1.0824,0.1678,inf,0.5898,1.5750,6.4491,0.0000,
8,(Gemma3:12b-QAT) - Llama3.3:70b,1.1443,0.1921,inf,0.5803,1.7082,5.9553,0.0000,
9,Gemma3:27b - (Gemma3:27b-QAT),1.5975,0.3570,inf,0.5497,2.6453,4.4752,0.0000,


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 12b. Exclusion pattern for Model 5 (manuscript Results, Metacognitive
#      Sensitivity paragraph) — is gamma-undefined status itself associated
#      with model/setsize/fb_exp/source_test?
#
#      NOTE: a full logistic regression on "excluded" is NOT fit here.
#      Gemma3:27b-QAT / no-feedback / imagined-source is deterministic
#      (400/400 traces excluded = 100%), which causes perfect separation
#      for that model x condition combination -- the same pathology
#      already flagged for Gemma3:27b-QAT's quasi-complete separation in
#      the Experiment 1 accuracy GLM (see Methods). A chi-square test of
#      independence is used instead, since it does not require converging
#      a model through a deterministic cell.
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## 12b. Exclusion Pattern (chi-square test, manuscript text check)"))

from scipy import stats

g_excl = df_gamma_src if "df_gamma_src" in dir() else None
gamma_fit_check = gamma_mod.calculate_gamma_across_groups(
    df, [sim_id] + grp_between + grp_within
)
gamma_fit_check["excluded"] = gamma_fit_check["gamma"].isna().astype(int)

ct = pd.crosstab(
    [gamma_fit_check["model"], gamma_fit_check["setsize"],
     gamma_fit_check["fb_exp"], gamma_fit_check["source_test"]],
    gamma_fit_check["excluded"],
)
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f"Omnibus chi-square (excluded ~ 24-cell condition): "
      f"chi2({dof}) = {chi2:.2f}, p = {p:.3e}")
print()

rate = (gamma_fit_check.groupby(["model", "fb_exp", "source_test"], observed=True)["excluded"]
        .mean() * 100).round(1)
print("Exclusion rate (%) by model x feedback x source, top 8:")
print(rate.sort_values(ascending=False).head(8))
print()

sub = gamma_fit_check[(gamma_fit_check["model"] == "Gemma3:27b-QAT")
                       & (gamma_fit_check["fb_exp"] == "False")
                       & (gamma_fit_check["source_test"] == "test:imagined")]
print("Gemma3:27b-QAT / no-feedback / imagined-source, excluded value counts:")
print(sub["excluded"].value_counts())

---
## 12b. Exclusion Pattern (chi-square test, manuscript text check)

Omnibus chi-square (excluded ~ 24-cell condition): chi2(47) = 3489.06, p = 0.000e+00

Exclusion rate (%) by model x feedback x source, top 8:
model           fb_exp  source_test
Gemma3:27b-QAT  False   test:imagined     100.0
                        test:perceived     99.2
                True    test:imagined      84.8
Gemma3:27b      True    test:perceived     81.2
Gemma3:12b      False   test:perceived     77.5
Gemma3:27b      True    test:imagined      76.2
Gemma3:27b-QAT  True    test:perceived     73.0
Gemma3:27b      False   test:perceived     68.5
Name: excluded, dtype: float64

Gemma3:27b-QAT / no-feedback / imagined-source, excluded value counts:
excluded
1    400
Name: count, dtype: int64


### Manuscript cross-reference — Gamma LMM source contrast

The output block above (Model 5 — Metacognitive γ) already computes
the emmeans source contrast. The key value reported in `main.tex` is:

```
test:perceived - test:imagined  estimate = 4.276  SE = 0.118  CI = [4.045, 4.507]  z = 36.29  p < .001
```

Reported in text as: pairwise contrast perceived − imagined = 4.28, SE = 0.12, p < .001

This appears in the cell above under **'Pairwise source contrast (source_test)'**.
The standalone call below reproduces it explicitly if re-running.

In [19]:
# Explicit source contrast — Gamma LMM (manuscript main.tex §Experiment 2 Metacognition)
# Reported in text: perceived - imagined = 4.28, SE = 0.12, p < .001
# Run this cell only after models_gamma has been fitted in Cell 25 above.

import rpy2.robjects as ro
ro.globalenv['gamma_int'] = models_gam['interactive'].r_model
ro.r("""
library(emmeans)
em_src_gam <- emmeans(gamma_int, ~ source_test)
cat('\n=== Gamma LMM: source emmeans contrast (perceived - imagined) ===\n')
print(contrast(em_src_gam, method = 'pairwise'))
""")


In [24]:
# Explicit source contrast — Gamma LMM (manuscript main.tex §Experiment 2 Metacognition)
# Reported in text: perceived - imagined = 4.28, SE = 0.12, p < .001
pairwise(models_gam['interactive'], 'source_test', label='Gamma LMM — source contrast')

**Gamma LMM — source contrast — Pairwise emmeans: source_test [fdr]**

,contrast,estimate,SE,df,asymp_LCL,asymp_UCL,z_ratio,p_value,sig
0,test:perceived - test:imagined,4.2761,0.1178,inf,4.0452,4.5071,36.289,0.0,


,contrast,estimate,SE,df,asymp_LCL,asymp_UCL,z_ratio,p_value,sig
0,test:perceived - test:imagined,4.276107,0.117835,inf,4.045155,4.50706,36.288971,2.415072e-288,


## Section 13: Model 5P — γ, perceived + rh_mean

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# 13. Model 5P — γ, perceived trials + rh_mean (trace-level)
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## Model 5P — Metacognitive γ (perceived + rh_mean)"))
display(Markdown(
    "γ computed on perceived-only trials per trace.  \n"
    "`rh_mean` = mean RH rate per trace (perceived trials) — between-trace covariate."
))

df_gamma_perc = gamma_mod.calculate_gamma_across_groups(
    df_perc, [sim_id] + grp_between
)
df_gamma_perc["f_gamma"] = fisher_z(df_gamma_perc["gamma"])

rh_trace = (
    df_perc.groupby(sim_id, observed=True)["read_hallucination"]
    .mean().reset_index()
    .rename(columns={"read_hallucination": "rh_mean"})
)
gamma_perc_fit = df_gamma_perc.copy()
if "trace" not in gamma_perc_fit.columns and sim_id in gamma_perc_fit.columns:
    gamma_perc_fit = gamma_perc_fit.rename(columns={sim_id: "trace"})
gamma_perc_fit = gamma_perc_fit.merge(rh_trace, on="trace", how="left")
for col in ["setsize","fb_exp","model"]:
    if col in gamma_perc_fit.columns:
        gamma_perc_fit[col] = gamma_perc_fit[col].astype(str)
gp_data = gamma_perc_fit[["f_gamma","setsize","fb_exp","model","rh_mean","trace"]].dropna()

_GAM_P_FE_INT = "setsize * fb_exp + model + rh_mean"
_GAM_P_FE_ADD = "setsize + fb_exp + model + rh_mean"

# γ (perceived) has ONE row per trace → (1|trace) is impossible (n_levels = n_obs).
# Use OLS (statsmodels) instead of lmer for this trace-level aggregated outcome.
display(Markdown("### γ (perceived) — OLS (trace-level aggregated, no RE)"))
import statsmodels.formula.api as smf

gp_data_ols = gp_data.copy()
# Ensure categoricals are strings for patsy
for col in ["setsize","fb_exp","model"]:
    gp_data_ols[col] = gp_data_ols[col].astype(str)

_gp_int_formula = f"f_gamma ~ {_GAM_P_FE_INT}"
_gp_add_formula = f"f_gamma ~ {_GAM_P_FE_ADD}"
_gp_nul_formula = "f_gamma ~ 1"

ols_int = smf.ols(_gp_int_formula, data=gp_data_ols).fit()
ols_add = smf.ols(_gp_add_formula, data=gp_data_ols).fit()
ols_nul = smf.ols(_gp_nul_formula, data=gp_data_ols).fit()

display(Markdown("**γ (perceived) — Null vs Additive (F-test)**"))
from scipy.stats import f as f_dist
def ols_lrt(m_red, m_full, label=""):
    df_diff = m_full.df_model - m_red.df_model
    f_stat  = ((m_red.ssr - m_full.ssr) / df_diff) / (m_full.ssr / m_full.df_resid)
    p_val   = 1 - f_dist.cdf(f_stat, df_diff, m_full.df_resid)
    sig = "***" if p_val < .001 else "**" if p_val < .01 else "*" if p_val < .05 else ""
    print(f"  {label}: F({int(df_diff)},{int(m_full.df_resid)})={f_stat:.3f}  p={p_val:.4e}  {sig}")
    print(f"  AIC null={m_red.aic:.1f}  AIC full={m_full.aic:.1f}")

ols_lrt(ols_nul, ols_add, "Null → Additive")
ols_lrt(ols_add, ols_int, "Additive → Interactive")

display(Markdown("**γ (perceived) — Interactive model coefficients**"))
coef_df = pd.DataFrame({
    "term":      ols_int.params.index,
    "estimate":  ols_int.params.values,
    "std_error": ols_int.bse.values,
    "t_stat":    ols_int.tvalues.values,
    "p_value":   ols_int.pvalues.values,
})
coef_df["sig"] = coef_df["p_value"].map(
    lambda p: "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "")
display(coef_df.round(4))
print(f"  R²={ols_int.rsquared:.4f}  Adj-R²={ols_int.rsquared_adj:.4f}  n={int(ols_int.nobs)}")

# Wrap in a dict so the summary table code doesn't crash
models_gam_p = {"null": None, "additive": None, "interactive": None, "int_reduced": None}

---
## Model 5P — Metacognitive γ (perceived + rh_mean)

γ computed on perceived-only trials per trace.  
`rh_mean` = mean RH rate per trace (perceived trials) — between-trace covariate.

### γ (perceived) — OLS (trace-level aggregated, no RE)

**γ (perceived) — Null vs Additive (F-test)**

  Null → Additive: F(8,2174)=66.465  p=1.1102e-16  ***
  AIC null=12451.0  AIC full=11989.3
  Additive → Interactive: F(1,2173)=34.099  p=6.0247e-09  ***
  AIC null=11989.3  AIC full=11957.3


**γ (perceived) — Interactive model coefficients**

,term,estimate,std_error,t_stat,p_value,sig
0,Intercept,2.3040,0.3366,6.8452,0.0000,***
1,setsize[T.40],-1.1141,0.2660,-4.1889,0.0000,***
2,fb_exp[T.True],-3.9855,0.2471,-16.1277,0.0000,***
3,model[T.Gemma3:12b-QAT],0.4155,0.2672,1.5553,0.1200,
4,model[T.Gemma3:27b],0.8412,0.3595,2.3400,0.0194,*
5,model[T.Gemma3:27b-QAT],-1.7946,0.4149,-4.3255,0.0000,***
6,model[T.Llama3.3:70b],3.1395,0.3716,8.4489,0.0000,***
7,model[T.Llama4:16x17b],1.9641,0.2919,6.7281,0.0000,***
8,setsize[T.40]:fb_exp[T.True],1.9720,0.3377,5.8395,0.0000,***
9,rh_mean,1.5486,0.3861,4.0112,0.0001,***


  R²=0.2089  Adj-R²=0.2057  n=2183


## Section 14: Model Comparison Summary Table

In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# 14. Model comparison summary table
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## 14. Model Comparison Summary"))

_all_models = [
    ("1.  RH (perceived)",      models_rh),
    ("2.  Accuracy",            models_acc),
    ("2P. Accuracy (perceived)", models_acc_p),
    ("3.  Rating",              models_rr),
    ("3P. Rating (perceived)",  models_rr_p),
    ("4.  Confidence",          models_conf),
    ("4P. Confidence (perc.)",  models_conf_p),
    ("5.  γ (all)",             models_gam),
    ("5P. γ (perceived)",       models_gam_p),
]

rows = []
for name, md in _all_models:
    for mtype in ("null", "additive", "interactive"):
        m = md.get(mtype)
        if m is None:
            continue
        rows.append({
            "Outcome":    name,
            "Model":      mtype,
            "AIC":        round(_aic(m), 1) if not np.isnan(_aic(m)) else "—",
            "BIC":        round(_bic(m), 1) if not np.isnan(_bic(m)) else "—",
            "n":          len(m.data) if hasattr(m, "data") and m.data is not None else "—",
        })

display(pd.DataFrame(rows))

---
## 14. Model Comparison Summary

,Outcome,Model,AIC,BIC,n
0,1. RH (perceived),null,37216.3,37233.3,36000
1,1. RH (perceived),additive,32966.1,33051.0,36000
2,1. RH (perceived),interactive,32947.4,33040.8,36000
3,2. Accuracy,null,98026.1,98044.5,72000
4,2. Accuracy,additive,93145.0,93255.2,72000
5,2. Accuracy,interactive,90282.8,90420.5,72000
6,2P. Accuracy (perceived),null,40906.5,40923.5,36000
7,2P. Accuracy (perceived),additive,39059.1,39161.0,36000
8,2P. Accuracy (perceived),interactive,39061.4,39171.8,36000
9,3. Rating,null,633845.1,633872.6,72000


## Section 15: Key Results Figure (6-panel)

In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# 15. Key Results Figure (6-panel)
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## 15. Key Results Figure"))

MODEL_ORDER  = sorted(df["model"].astype(str).unique())
MODEL_LABELS = [m.replace("-", "-\n") for m in MODEL_ORDER]
FB_VALS      = ["True", "False"]
FB_LABELS    = {"True": "Feedback Present", "False": "Feedback Absent"}
x, bar_w     = np.arange(len(MODEL_ORDER)), 0.38

df_gamma_sim = gamma_mod.calculate_gamma_across_groups(df, [sim_id] + grp_between)
df_gamma_sim["f_gamma"] = fisher_z(df_gamma_sim["gamma"])

rh_means   = df_perc.groupby(["model","fb_exp"], observed=True)["read_hallucination"].mean().to_dict()
acc_means  = df.groupby(["model","fb_exp"],      observed=True)["accuracy"].mean().to_dict()
src_cats   = df["source_test"].unique().tolist()
perc_src   = next(s for s in src_cats if "perceived" in s)
gam_all_d  = df_gamma_sim.groupby(["model","fb_exp"], observed=True)["f_gamma"].mean().to_dict()
gam_per_d  = (df_gamma_src[df_gamma_src["source_test"] == perc_src]
              .groupby(["model","fb_exp"], observed=True)["f_gamma"].mean().to_dict())
rr_m = df.groupby(["model","source_test","fb_exp"], observed=True)["rating_cen"].mean().to_dict()
rr_s = df.groupby(["model","source_test","fb_exp"], observed=True)["rating_cen"].sem().to_dict()
conf_levels = sorted(df["confidence_num"].dropna().astype(int).unique())


def _bar(ax, d, ylabel="", ylim=(0,1)):
    for fi, fb in enumerate(FB_VALS):
        vals = [d.get((m, fb), np.nan) for m in MODEL_ORDER]
        ax.bar(x + (fi-.5)*bar_w, vals, width=bar_w,
               color=FB_PALETTE[fb], label=FB_LABELS[fb],
               alpha=0.88, edgecolor="white", lw=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_LABELS, fontsize=9, fontweight="bold",
                       rotation=45, ha="right", rotation_mode="anchor")
    ax.set_xlim(-.6, len(MODEL_ORDER)-.4)
    ax.set_ylabel(ylabel, fontsize=11, fontweight="bold")
    ax.set_ylim(*ylim)


def _ann(ax, letter, title):
    ax.text(-.15, 1.08, letter, transform=ax.transAxes,
            fontsize=14, fontweight="bold", va="top")
    ax.set_title(title, pad=8, fontsize=12, fontweight="bold")


fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=.65, wspace=.42,
                        left=.07, right=.97, top=.92, bottom=.14)

ax_a = fig.add_subplot(gs[0,0])
_bar(ax_a, rh_means, "RH Rate (perceived only)")
ax_a.legend(title="Feedback", fontsize=8, loc="upper left")
_ann(ax_a, "A", "Reading Hallucination")

ax_b = fig.add_subplot(gs[0,1])
_bar(ax_b, acc_means, "Recognition Accuracy")
ax_b.legend(title="Feedback", fontsize=8, loc="lower right")
_ann(ax_b, "B", "Recognition Accuracy")

ax_c = fig.add_subplot(gs[0,2])
COLS = sns.color_palette("tab10", n_colors=len(MODEL_ORDER))
for mi, mdl in enumerate(MODEL_ORDER):
    sub = df[(df["model"]==mdl) & (df["source_test"]==perc_src)]
    if sub.empty:
        continue
    p = (sub["confidence_num"].dropna().astype(int)
         .value_counts(normalize=True).reindex(conf_levels, fill_value=0.0))
    ax_c.plot(p.index, p.values, color=COLS[mi], lw=1.4, alpha=.75,
              label=mdl.replace("-","-\n"))
ax_c.set_xlabel("Confidence Level", fontsize=11, fontweight="bold")
ax_c.set_ylabel("Proportion",       fontsize=11, fontweight="bold")
ax_c.set_xticks(conf_levels)
ax_c.legend(ncol=2, fontsize=7)
_ann(ax_c, "C", "Confidence Distribution")

ax_d = fig.add_subplot(gs[1,0])
_bar(ax_d, gam_all_d, "Fisher's Z (γ)", ylim=(-2,4))
ax_d.axhline(0, color="black", lw=.8, ls="--", alpha=.5)
ax_d.legend(title="Feedback", fontsize=8)
_ann(ax_d, "D", "Metacognitive γ (All)")

ax_e = fig.add_subplot(gs[1,1])
_bar(ax_e, gam_per_d, "Fisher's Z (γ)", ylim=(-2,5))
ax_e.axhline(0, color="black", lw=.8, ls="--", alpha=.5)
ax_e.legend(title="Feedback", fontsize=8)
_ann(ax_e, "E", "Metacognitive γ (Perceived)")

ax_f = fig.add_subplot(gs[1,2])
ls_map = ["-","--"]
for si, src in enumerate(src_cats):
    for fb in FB_VALS:
        means = [rr_m.get((ml,src,fb), np.nan) for ml in MODEL_ORDER]
        ses   = [rr_s.get((ml,src,fb), 0.)     for ml in MODEL_ORDER]
        ax_f.errorbar(x, means, yerr=ses, color=FB_PALETTE[fb], ls=ls_map[si],
                      lw=2., marker="o", ms=5, capsize=3, alpha=.88,
                      label=f"{FB_LABELS[fb]} · {src}")
ax_f.set_xticks(x)
ax_f.set_xticklabels(MODEL_LABELS, fontsize=9, fontweight="bold",
                     rotation=45, ha="right", rotation_mode="anchor")
ax_f.set_xlim(-.6, len(MODEL_ORDER)-.4)
ax_f.set_ylabel("Rating (centred, Mean ± SE)", fontsize=11, fontweight="bold")
ax_f.legend(fontsize=7, loc="lower right")
_ann(ax_f, "F", "Relatedness Rating")

fig.suptitle("Experiment 2 — Key Results (v3)", fontsize=14, fontweight="bold", y=.98)
plt.savefig("exp2_key_results_v3.pdf", dpi=300, bbox_inches="tight")
plt.savefig("exp2_key_results_v3.png", dpi=300, bbox_inches="tight")
plt.close('all')
print("Figure saved.")

---
## 15. Key Results Figure

Figure saved.


## Section 16: Summary: Preprocessing & Model Structure

In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# 16. Summary table
# ─────────────────────────────────────────────────────────────────────────────
display(Markdown("---\n## 16. Summary"))
display(Markdown("""
**Preprocessing:**
- `order_c = order − 1`  (0 = first presentation, 1 = second)
- `rating_cen = rating − grand_mean`  (centred, not z-scored)

**Model structure per outcome:**

| Step | Comparison | What is tested |
|---|---|---|
| 1 | RE: `(1+source_test\\|trace)` vs `(1\\|trace)` | Random-slope necessity |
| 2 | FE: null → additive | Any fixed effects improve fit |
| 3 | FE: additive → interactive | 2-way interactions needed |

| Model | Trials | Family | Key fixed effects |
|---|---|---|---|
| 1. RH | perceived | Binomial | setsize×fb_exp, model, order_c |
| 2. Accuracy | all | Binomial | source_test, setsize, fb_exp, 2-ways, model, order_c, rating_cen |
| 2P. Accuracy | perceived | Binomial | setsize×fb_exp, model, order_c, rating_cen, **read_hallucination** |
| 3. Rating | all | Gaussian | source_test, setsize, fb_exp, 2-ways, model, order_c |
| 3P. Rating | perceived | Gaussian | setsize×fb_exp, model, order_c, **read_hallucination** |
| 4. Confidence | all | Gaussian | accuracy, source_test, setsize, fb_exp, 2-ways, model, order_c, rating_cen |
| 4P. Confidence | perceived | Gaussian | accuracy, setsize×fb_exp, model, order_c, rating_cen, **read_hallucination** |
| 5. γ | all | Gaussian | source_test, setsize, fb_exp, 2-ways, model |
| 5P. γ | perceived | Gaussian | setsize×fb_exp, model, **rh_mean** |

**Post-hoc contrasts (FDR-corrected):**
H1 source_test · H2 fb_exp · H3 source_test×fb_exp · H4 model · covariates (slopes)
"""))
print("Analysis complete.")

---
## 16. Summary


**Preprocessing:**
- `order_c = order − 1`  (0 = first presentation, 1 = second)
- `rating_cen = rating − grand_mean`  (centred, not z-scored)

**Model structure per outcome:**

| Step | Comparison | What is tested |
|---|---|---|
| 1 | RE: `(1+source_test\|trace)` vs `(1\|trace)` | Random-slope necessity |
| 2 | FE: null → additive | Any fixed effects improve fit |
| 3 | FE: additive → interactive | 2-way interactions needed |

| Model | Trials | Family | Key fixed effects |
|---|---|---|---|
| 1. RH | perceived | Binomial | setsize×fb_exp, model, order_c |
| 2. Accuracy | all | Binomial | source_test, setsize, fb_exp, 2-ways, model, order_c, rating_cen |
| 2P. Accuracy | perceived | Binomial | setsize×fb_exp, model, order_c, rating_cen, **read_hallucination** |
| 3. Rating | all | Gaussian | source_test, setsize, fb_exp, 2-ways, model, order_c |
| 3P. Rating | perceived | Gaussian | setsize×fb_exp, model, order_c, **read_hallucination** |
| 4. Confidence | all | Gaussian | accuracy, source_test, setsize, fb_exp, 2-ways, model, order_c, rating_cen |
| 4P. Confidence | perceived | Gaussian | accuracy, setsize×fb_exp, model, order_c, rating_cen, **read_hallucination** |
| 5. γ | all | Gaussian | source_test, setsize, fb_exp, 2-ways, model |
| 5P. γ | perceived | Gaussian | setsize×fb_exp, model, **rh_mean** |

**Post-hoc contrasts (FDR-corrected):**
H1 source_test · H2 fb_exp · H3 source_test×fb_exp · H4 model · covariates (slopes)


Analysis complete.
